In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from __future__ import annotations

In [ ]:
import anndata as ad
from adjustText import adjust_text

from cellassign import assign_cats

from cellbender.remove_background.downstream import load_anndata_from_input_and_output as load_anndata_cellbender

import cellrank as cr
from cellrank.estimators import GPCCA

import doubletdetection

from fa2 import ForceAtlas2

import gc

import harmonypy as hm

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager, rcParams

import networkx as nx

import numpy as np

import palantir

import pandas as pd

import phate

import plotly.express as px

from pybiomart import Server

import re 

from rpy2.robjects import globalenv
from rpy2.robjects import pandas2ri

import scanpy as sc
import scanpy.external as sce

import scFates as scf

from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection

import scipy.sparse as sp
from scipy.sparse import csr_matrix, issparse

import scvelo as scv

import seaborn as sns

from sklearn.decomposition import PCA

import triku as tk
import os, subprocess

In [ ]:
import sys

sys.path.append('..')

from pyfuncs.io import load_full_adata, add_ensembl_ids, save_deg_to_excel_simple
from pyfuncs.dropletQC import classify_empty_and_damaged
from pyfuncs.general import preprocessing_adata_sub
from pyfuncs.plot_functions import magma, set_plotting_style, savefig, plot_volcano
from pyfuncs.common_vars import BASE_DIR, SEED, CELLBENDER_FIXED_ARGS
set_plotting_style()

In [ ]:
from pyfuncs.qc import  MT_CONTIG_MOUSE_REFSEQ, compute_qc_metrics, add_droplet_qc, flag_doublets, qc_embedding, plot_qc_overview, nf_band_report, ambient_top_genes, apply_qc_flags, qc_summary
from pyfuncs.normalization import concat_samples, preliminary_clusters, scran_size_factors, apply_size_factors, compare_normalizations, size_factor_report
from pyfuncs.processing import select_hvgs_preliminary, build_embeddings, select_hvgs_triku, soup_vs_hvg_report, harmony_merge_report
from pyfuncs.characterization import subset_and_reprocess, check_marker_dict, population_composition, reprocess_in_place
from pyfuncs.cell_types import DICT_MARKERS_MAJOR_POPULATIONS, DICT_MARKERS_FAP, DICT_MARKERS_KRANOCYTE, DICT_MARKERS_SATELLITE, DICT_MARKERS_TENO, DICT_RENAMING, PALETTE_CELL_TYPE
from pyfuncs.ontology import build_background, run_goea
from pyfuncs.degs import depth_report, deg_by_group, deg_sets, plot_upset, lfc_concordance, query_sets

In [ ]:
from pyfuncs.io import ambient_fraction_per_gene
from pyfuncs.trajectory import (
    normalize_velocyto_layers,
    run_velocity,
    paga_report,
    paga_stability,
    plot_velocity,
    run_phate
)

from pyfuncs.interactions import (mouse_to_human, prepare_cpdb_input,
                                  download_cpdb_database, run_cpdb_statistical,
                                  filter_interactions, plot_interactions, liana_resources, resource_coverage,
                                  run_liana, filter_interactions, filter_liana, annotate_liana, delta_liana)

In [ ]:
from datetime import date
TODAY = str(date.today())

DATA_DIR = f"{BASE_DIR}/data/public_scRNAseq/datasets/southerland_2023_GSE227075"
FIG_DIR = f"{BASE_DIR}/figures/{TODAY}/"
RESULTS_DIR = f"{BASE_DIR}/results/{TODAY}/"

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

REFERENCE_DIR = f"{BASE_DIR}/data/public_scRNAseq/common/reference"
GTF = f"{REFERENCE_DIR}/GRCm39_NCBI_GCF_000001635.27/GCF_000001635.27_GRCm39_genomic.gtf"

SEED = 10

# Adata loading

In [ ]:
adata = sc.read(f"{DATA_DIR}/processed_adatas/SU_southerland_processed.h5ad")
adata_FAP = sc.read(f"{DATA_DIR}/processed_adatas/SU_southerland_processed_FAPs.h5ad")

In [ ]:
sc.pl.umap(adata, color="cell_type")

In [ ]:
sc.pl.umap(adata_FAP, color=["condition", "gsm", "cell_type"], frameon=False)

In [ ]:
_ = depth_report(adata_FAP, group_key="cell_type", condition_key="condition")

In [ ]:
adata_FAP.obs[["condition", "cell_type"]].value_counts().sort_index().reset_index().pivot(columns="condition", index="cell_type", values="count")

# DEGs between sham (CTRL) and HLI_14d

In [ ]:
dict_DEGs = deg_by_group(adata_FAP, group_key="cell_type", condition_key="condition", 
             reference="C57BL6_sham", test="C57BL6_HLI_day1", min_cells=15)


### Analysis of upregulated genes: volcano plot

In [ ]:
list_DEGs = []

for population in [f"FAP.{i}" for i in [1, 2, 3, 4, 5]]:
    print(population)
    adata_FAP_pop = adata_FAP[adata_FAP.obs['cell_type'] == population]
    sc.tl.rank_genes_groups(adata_FAP_pop, groupby="condition", method="wilcoxon", pts=True)
    df = plot_volcano(adata_FAP_pop, cluster="C57BL6_HLI_day1", bottomn=0, topn=50, xlim=(-0.1, 7), return_df=True, pval_threshold=1e-3, lfc_threshold=1)
    savefig(plt.gcf(), filename=f"3SU_VolcanoPlot_FAP_HLIshamvs1d_up", fig_dir=FIG_DIR)

    df = df.sort_values("pvalxlfc", ascending=False).reset_index(drop=True)
    df["population"] = population
    list_DEGs.append(df)

df_concat = pd.concat(list_DEGs)
save_deg_to_excel_simple(df_concat, f"{RESULTS_DIR}/3SU_DEGs_FAP_HLIshamvs1d.xlsx", group_col='population',
    cols=("gene", "adj_pval", "logfoldchanges", "pvalxlfc"))

### Analysis of upregulated genes: upsetplot + GOEA

In [ ]:
background_FAP = build_background(adata_FAP, group_key="cell_type")

In [ ]:
dict_DEGs["FAP.1"][dict_DEGs["FAP.1"]["names"] == "Ccl2"]

In [ ]:
up   = deg_sets(dict_DEGs, n_top=150, direction="up", min_delta_pct=0.1)

fig, upset_stats = plot_upset(up)
savefig(fig=fig, filename=f"3SU_UPSetPlot_FAP_HLIshamvs1d_up", fig_dir=FIG_DIR)

In [ ]:
dict_degs_FAP_HLIshamvs1d_up = {"FAP.1-exclusive": query_sets(up, ["FAP.1"], res=dict_DEGs, verbose=False)["gen"].values.tolist(),
                                "FAP.1": query_sets(up, ["FAP.1"], res=dict_DEGs, verbose=False, exact=False)["gen"].values.tolist(),
                       "FAP.2-exclusive": query_sets(up, ["FAP.2"], res=dict_DEGs, verbose=False)["gen"].values.tolist(),
                       "FAP.2": query_sets(up, ["FAP.2"], res=dict_DEGs, verbose=False, exact=False)["gen"].values.tolist(),
                       "FAP.3-exclusive": query_sets(up, ["FAP.3"], res=dict_DEGs, verbose=False)["gen"].values.tolist(),
                       "FAP.3": query_sets(up, ["FAP.3"], res=dict_DEGs, verbose=False, exact=False)["gen"].values.tolist(),
                       "FAP.4-exclusive": query_sets(up, ["FAP.4"], res=dict_DEGs, verbose=False)["gen"].values.tolist(),
                       "FAP.4": query_sets(up, ["FAP.4"], res=dict_DEGs, verbose=False, exact=False)["gen"].values.tolist(),
                       "pairs": query_sets(up, n_populations=2, res=dict_DEGs, verbose=False)["gen"].values.tolist(), 
                       "common": query_sets(up, n_populations=3, res=dict_DEGs, verbose=False)["gen"].values.tolist() + \
                                 query_sets(up, n_populations=4, res=dict_DEGs, verbose=False)["gen"].values.tolist() + \
                                 query_sets(up, n_populations=5, res=dict_DEGs, verbose=False)["gen"].values.tolist(), 
                    }

GOEA_FAP_HLIshamvs1d_up = run_goea(gene_dict = dict_degs_FAP_HLIshamvs1d_up, background=background_FAP)
GOEA_FAP_HLIshamvs1d_up["log_padj"] = -np.log10(GOEA_FAP_HLIshamvs1d_up["padj_global"])


In [ ]:
for poblacion in dict_degs_FAP_HLIshamvs1d_up.keys():
    print(poblacion)
    df_sub = GOEA_FAP_HLIshamvs1d_up[GOEA_FAP_HLIshamvs1d_up["poblacion"] == poblacion]
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    sns.barplot(data = df_sub.iloc[:20], y="termino", x="log_padj", ax=ax)
    plt.axvline(-np.log10(0.05), c="#bc0000")
    plt.title(f"Top GO terms for {poblacion}")
    savefig(fig=fig, filename=f"3SU_GO-terms_FAP_HLIshamvs1d_{poblacion}_up", fig_dir=FIG_DIR)
    plt.show()

save_deg_to_excel_simple(GOEA_FAP_HLIshamvs1d_up, f"{RESULTS_DIR}/3SU_df_GOEA_FAP_HLIshamvs1d_up.xlsx", group_col='poblacion',
    cols=('libreria','termino','solapamiento','odds_ratio', "pval", "padj_global", "genes"))

### Analysis of downregulated genes: volcano plot

In [ ]:
list_DEGs = []

for population in [f"FAP.{i}" for i in [1, 2, 3, 4, 5]]:
    print(population)
    adata_FAP_pop = adata_FAP[adata_FAP.obs['cell_type'] == population]
    sc.tl.rank_genes_groups(adata_FAP_pop, groupby="condition", method="wilcoxon", pts=True)
    df = plot_volcano(adata_FAP_pop, cluster="C57BL6_HLI_day1", bottomn=30, topn=0, xlim=(-5, 0.), return_df=True, pval_threshold=1e-3, lfc_threshold=1, plot_positive_only=False)
    savefig(plt.gcf(), filename=f"3SU_VolcanoPlot_FAP_HLIshamvs1d_down", fig_dir=FIG_DIR)

    df = df.sort_values("pvalxlfc", ascending=False).reset_index(drop=True)
    df["population"] = population
    list_DEGs.append(df)

### Analysis of downregulated genes: upsetplot + GOEA

In [ ]:
down   = deg_sets(dict_DEGs, n_top=150, direction="down", min_delta_pct=0.01
                )
fig, upset_stats = plot_upset(down)
savefig(fig=fig, filename=f"3SU_UPSetPlot_FAP_HLIshamvs1d_down", fig_dir=FIG_DIR)

In [ ]:
dict_degs_FAP_HLIshamvs1d_down = {"FAP.1-exclusive": query_sets(down, ["FAP.1"], res=dict_DEGs, verbose=False)["gen"].values.tolist(),
                                "FAP.1": query_sets(down, ["FAP.1"], res=dict_DEGs, verbose=False, exact=False)["gen"].values.tolist(),
                       "FAP.2-exclusive": query_sets(down, ["FAP.2"], res=dict_DEGs, verbose=False)["gen"].values.tolist(),
                       "FAP.2": query_sets(down, ["FAP.2"], res=dict_DEGs, verbose=False, exact=False)["gen"].values.tolist(),
                       "FAP.3-exclusive": query_sets(down, ["FAP.3"], res=dict_DEGs, verbose=False)["gen"].values.tolist(),
                       "FAP.3": query_sets(down, ["FAP.3"], res=dict_DEGs, verbose=False, exact=False)["gen"].values.tolist(),
                       "FAP.4-exclusive": query_sets(down, ["FAP.4"], res=dict_DEGs, verbose=False)["gen"].values.tolist(),
                       "FAP.4": query_sets(down, ["FAP.4"], res=dict_DEGs, verbose=False, exact=False)["gen"].values.tolist(),
                       "pairs": query_sets(down, n_populations=2, res=dict_DEGs, verbose=False)["gen"].values.tolist(), 
                       "common (>=3)": query_sets(down, n_populations=3, res=dict_DEGs, verbose=False)["gen"].values.tolist() + \
                                 query_sets(down, n_populations=4, res=dict_DEGs, verbose=False)["gen"].values.tolist() + \
                                 query_sets(down, n_populations=5, res=dict_DEGs, verbose=False)["gen"].values.tolist(), 
                    }

GOEA_FAP_HLIshamvs1d_down = run_goea(gene_dict = dict_degs_FAP_HLIshamvs1d_down, background=background_FAP)
GOEA_FAP_HLIshamvs1d_down["log_padj"] = -np.log10(GOEA_FAP_HLIshamvs1d_down["padj_global"])

In [ ]:
for poblacion in dict_degs_FAP_HLIshamvs1d_down.keys():
    print(poblacion)
    df_sub = GOEA_FAP_HLIshamvs1d_down[GOEA_FAP_HLIshamvs1d_down["poblacion"] == poblacion]
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    sns.barplot(data = df_sub.iloc[:20], y="termino", x="log_padj", ax=ax)
    plt.axvline(-np.log10(0.05), c="#bc0000")
    plt.title(f"Top GO terms for {poblacion}")
    savefig(fig=fig, filename=f"3SU_GO-terms_FAP_HLIshamvs1d_{poblacion}_down", fig_dir=FIG_DIR)
    plt.show()

save_deg_to_excel_simple(GOEA_FAP_HLIshamvs1d_down, f"{RESULTS_DIR}/3SU_df_GOEA_FAP_HLIshamvs1d_down.xlsx", group_col='poblacion',
    cols=('libreria','termino','solapamiento','odds_ratio', "pval", "padj_global", "genes"))

# Presence of interleukin (IL6, IL1) regulation
We are going to check at a general and FAP level the presence/expression of
* IL6-related markers:
    * Il6 - the interleukin
    * Il6ra + Il6st [gp130] the two-part receptor
* IL1B-related markers:
    * Il1b - the interleukin
    * Il1r1 - the main receptor
    * Myd88 - the main signal transducer

In [ ]:
IL6_MARKERS_EXTENDED = ["Il6", "Il6ra", "Il6st"] 

sc.pl.umap(adata_FAP, color=["cell_type"] + [i for i in IL6_MARKERS_EXTENDED if i in adata_FAP.var_names], cmap=magma, frameon=False, vmax=4)

sc.pl.umap(adata_FAP[adata_FAP.obs["condition"] == "C57BL6_sham"], color=["cell_type"] + [i for i in IL6_MARKERS_EXTENDED if i in adata_FAP.var_names], cmap=magma, frameon=False, vmax=4, show=False)
plt.gcf().suptitle("C57BL6_sham")

sc.pl.umap(adata_FAP[adata_FAP.obs["condition"] == "C57BL6_HLI_day1"], color=["cell_type"] + [i for i in IL6_MARKERS_EXTENDED   if i in adata_FAP.var_names], cmap=magma, frameon=False, vmax=4, show=False)
plt.gcf().suptitle("C57BL6_HLI_day1")


In [ ]:
IL1B_MARKERS_EXTENDED = ["Il1b", "Il1r1", "Myd88"] + ["Il1rap", "Il1r2"]

sc.pl.umap(adata_FAP, color=["cell_type"] + [i for i in IL1B_MARKERS_EXTENDED if i in adata_FAP.var_names], cmap=magma, frameon=False, vmax=4)

sc.pl.umap(adata_FAP[adata_FAP.obs["condition"] == "C57BL6_sham"], color=["cell_type"] + [i for i in IL1B_MARKERS_EXTENDED if i in adata_FAP.var_names], cmap=magma, frameon=False, vmax=4, show=False)
plt.gcf().suptitle("C57BL6_sham")

sc.pl.umap(adata_FAP[adata_FAP.obs["condition"] == "C57BL6_HLI_day1"], color=["cell_type"] + [i for i in IL1B_MARKERS_EXTENDED if i in adata_FAP.var_names], cmap=magma, frameon=False, vmax=4, show=False)
plt.gcf().suptitle("C57BL6_HLI_day1")


### Running LIANA

In [ ]:
adata_FAP_C57BL6sham = adata_FAP[adata_FAP.obs["condition"] == "C57BL6_sham"]
adata_FAP_C57BL6_HLI_day1 = adata_FAP[adata_FAP.obs["condition"] == "C57BL6_HLI_day1"]

In [ ]:
liana_FAP  = annotate_liana(run_liana(adata_FAP, "cell_type", n_perms=25000, n_jobs=20), adata_FAP, "cell_type")

In [ ]:
liana_FAP_C57BL6sham = annotate_liana(run_liana(adata_FAP_C57BL6sham, "cell_type", n_perms=25000, n_jobs=20), adata_FAP_C57BL6sham, "cell_type")
liana_FAP_C57BL6_HLI_day1 = annotate_liana(run_liana(adata_FAP_C57BL6_HLI_day1, "cell_type", n_perms=25000, n_jobs=20), adata_FAP_C57BL6_HLI_day1, "cell_type")

#### Sub analysis of Il6-related markers

In [ ]:
liana_FAP_il6  = filter_liana(liana_FAP, IL6_MARKERS_EXTENDED)
liana_FAP_il6

In [ ]:
delta_il6 = delta_liana(liana_FAP_C57BL6sham, liana_FAP_C57BL6_HLI_day1, seleccion=liana_FAP_il6, nombres=("C57BL6_sham", "C57BL6_HLI_day1"),
                adatas=(adata_FAP_C57BL6sham, adata_FAP_C57BL6_HLI_day1), groupby="cell_type")

In [ ]:
delta_il6.sort_values(by="d_lr_logfc")

In [ ]:
MARKERS_IL11 = ["Il11", "Il11ra1", "Il6ra", "Il6", "Musk"]

sc.pl.umap(adata_FAP, color=["cell_type"] + [i for i in MARKERS_IL11 if i in adata_FAP.var_names], cmap=magma, frameon=False, vmax=4)

print("SHAM")
sc.pl.umap(adata_FAP[adata_FAP.obs["condition"] == "C57BL6_sham"], color=["cell_type"] + [i for i in MARKERS_IL11 if i in adata_FAP.var_names], cmap=magma, frameon=False, vmax=4, show=False)
plt.gcf().suptitle("C57BL6_sham")
plt.show()

print("HLI")
sc.pl.umap(adata_FAP[adata_FAP.obs["condition"] == "C57BL6_HLI_day1"], color=["cell_type"] + [i for i in MARKERS_IL11 if i in adata_FAP.var_names], cmap=magma, frameon=False, vmax=4, show=False)
plt.gcf().suptitle("C57BL6_HLI_day1")

#### Sub analysis of Il1b-related markers

In [ ]:
liana_FAP_il1b  = filter_liana(liana_FAP, IL1B_MARKERS_EXTENDED)
liana_FAP_il1b

In [ ]:
delta_il1b = delta_liana(liana_FAP_C57BL6sham, liana_FAP_C57BL6_HLI_day1, seleccion=liana_FAP_il1b, nombres=("C57BL6_sham", "C57BL6_HLI_day1"),
                adatas=(adata_FAP_C57BL6sham, adata_FAP_C57BL6_HLI_day1), groupby="cell_type")

In [ ]:
delta_il1b.sort_values(by="d_lr_logfc")

In [ ]:
MARKERS = ["Il11"]

sc.pl.umap(adata_FAP, color=["cell_type"] + [i for i in MARKERS if i in adata_FAP.var_names], cmap=magma, frameon=False, vmax=4)

sc.pl.umap(adata_FAP[adata_FAP.obs["condition"] == "C57BL6_sham"], color=["cell_type"] + [i for i in MARKERS if i in adata_FAP.var_names], cmap=magma, frameon=False, vmax=4, show=False)
plt.gcf().suptitle("C57BL6_sham")

sc.pl.umap(adata_FAP[adata_FAP.obs["condition"] == "C57BL6_HLI_day1"], color=["cell_type"] + [i for i in MARKERS if i in adata_FAP.var_names], cmap=magma, frameon=False, vmax=4, show=False)
plt.gcf().suptitle("C57BL6_HLI_day1")

In [ ]:
dl = delta_liana(liana_FAP_C57BL6sham, liana_FAP_C57BL6_HLI_day1)
dl = dl.dropna().sort_values("d_lr_logfc")
dl.head(20)

In [ ]:
dl.tail(20)

In [ ]:
MARKERS = ["Cxcl10", "Cxcl12", "Col8a1", "Sdc3", "Ptn", "Apoe", "Ldlr", "Angptl4"]

print("C57BL6_sham")
sc.pl.umap(adata_FAP[adata_FAP.obs["condition"] == "C57BL6_sham"], color=["cell_type"] + [i for i in MARKERS if i in adata_FAP.var_names], cmap=magma, frameon=False, vmax=4, show=False)
plt.gcf().suptitle("C57BL6_sham")
plt.show() 

print("C57BL6_HLI_day1")
sc.pl.umap(adata_FAP[adata_FAP.obs["condition"] == "C57BL6_HLI_day1"], color=["cell_type"] + [i for i in MARKERS if i in adata_FAP.var_names], cmap=magma, frameon=False, vmax=4, show=False)
plt.gcf().suptitle("C57BL6_HLI_day1")

In [ ]:
dl[(dl["source"].isin(["FAP.4"])) |  (dl["target"].isin(["FAP.4"])) ].head(10)

In [ ]:
dl[(dl["source"].isin(["FAP.4"])) |  (dl["target"].isin(["FAP.4"])) ].tail(10)

In [ ]:
MARKERS = ["Cxcl10", "Col8a1", "Itga1", "Itgb1", "Sdc4"]

print("C57BL6_sham")
sc.pl.umap(adata_FAP[adata_FAP.obs["condition"] == "C57BL6_sham"], color=["cell_type"] + [i for i in MARKERS if i in adata_FAP.var_names], cmap=magma, frameon=False, vmax=4, show=False)
plt.gcf().suptitle("C57BL6_sham")
plt.show() 

print("C57BL6_HLI_day1")
sc.pl.umap(adata_FAP[adata_FAP.obs["condition"] == "C57BL6_HLI_day1"], color=["cell_type"] + [i for i in MARKERS if i in adata_FAP.var_names], cmap=magma, frameon=False, vmax=4, show=False)
plt.gcf().suptitle("C57BL6_HLI_day1")